<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-2511-IMAGE-EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q torchsde

!apt install aria2 -qqy

# Create folders
!mkdir -p /content/ComfyUI/models/diffusion_models
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/loras

# Download Diffusion model (FP8 mixed ~20.5GB - recommended)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image-Edit_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors \
  -d /content/ComfyUI/models/diffusion_models -o qwen_image_edit_2511_fp8mixed.safetensors

# Download Text Encoder
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors \
  -d /content/ComfyUI/models/text_encoders -o qwen_2.5_vl_7b_fp8_scaled.safetensors

# Download VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

# Download Lightning LoRA (optional, for 4-step speed)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/lightx2v/Qwen-Image-Edit-2511-Lightning/resolve/main/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors \
  -d /content/ComfyUI/models/loras -o Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors

# Check downloads
!ls -lh /content/ComfyUI/models/diffusion_models/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/
!ls -lh /content/ComfyUI/models/loras/

/content
fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
 *** Download Progress Summary as of Wed Nov 11 03:24:45 2026 *** 
=
[#fe363b 11GiB/19GiB(61%) CN:16 DL:181MiB ETA:41s]
FILE: /content/ComfyUI/models/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
fe363b|OK  |   174MiB/s|/content/ComfyUI/models/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors

Status Legend:
(OK):download completed.
 *** Download Progress Summary as of Wed Nov 11 03:26:38 2026 *** 
=
[#0af5ca 8.7GiB/8.7GiB(99%) CN:1 DL:138MiB]
FILE: /content/ComfyUI/models/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+===========

In [ ]:
%cd /content/ComfyUI

import torch
import logging
import warnings

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
LoraLoaderModelOnly = NODE_CLASS_MAPPINGS["LoraLoaderModelOnly"]()

with torch.inference_mode():
    # Load UNET
    unet = UNETLoader.load_unet("qwen_image_edit_2511_fp8mixed.safetensors", "default")[0]

    # Load CLIP (text encoder)
    clip = CLIPLoader.load_clip("qwen_2.5_vl_7b_fp8_scaled.safetensors", "qwen_image")[0]

    # Load VAE
    vae = VAELoader.load_vae("qwen_image_vae.safetensors")[0]

    # Optional: Apply Lightning LoRA for 4-step speed
    # unet = LoraLoaderModelOnly.load_lora_model_only(unet, "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors", 1.0)[0]

print("Qwen-Image-Edit-2511 loaded successfully.")

In [ ]:
%cd /content/ComfyUI

import random
from PIL import Image
from IPython.display import display

CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()

positive = "photorealistic portrait of a woman, natural skin texture, detailed face, high quality"
negative = ""

with torch.inference_mode():
    pos = CLIPTextEncode.encode(clip, positive)[0]
    neg = ConditioningZeroOut.zero_out(pos)[0]
    latent = EmptyLatentImage.generate(1024, 1024, 1)[0]

    # Without Lightning: 20-40 steps, CFG 4
    # With Lightning: 4 steps, CFG 1
    samples = KSampler.sample(
        unet,
        random.randint(1, 2**32),
        20,           # steps
        4.0,          # cfg
        "euler",
        "simple",
        pos,
        neg,
        latent,
        1.0
    )[0]

    image = VAEDecode.decode(vae, samples)[0]
    img = Image.fromarray((image.cpu().numpy().clip(0,1)*255).astype("uint8"))
    display(img)

In [ ]:
%cd /content/ComfyUI

import threading
import time
import subprocess
import re
import base64
import io
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    positive: str = "photorealistic portrait of a woman"
    negative: str = ""
    steps: int = 20
    cfg: float = 4.0
    width: int = 1024
    height: int = 1024
    seed: int = -1

@app.options("/{path:path}")
async def options_handler(path: str):
    return {"status": "ok"}

@app.post("/generate")
def generate(req: GenerateRequest):
    try:
        seed = req.seed if req.seed != -1 else random.randint(1, 2**32)

        with torch.inference_mode():
            pos = CLIPTextEncode.encode(clip, req.positive)[0]
            neg = ConditioningZeroOut.zero_out(pos)[0]
            latent = EmptyLatentImage.generate(req.width, req.height, 1)[0]
            samples = KSampler.sample(
                unet, seed, req.steps, req.cfg, "euler", "simple", pos, neg, latent, 1.0
            )[0]
            image = VAEDecode.decode(vae, samples)[0]

        img = Image.fromarray((image.cpu().numpy().clip(0,1)*255).astype("uint8"))
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        img_base64 = base64.b64encode(buffer.getvalue()).decode()

        return {"image": img_base64, "seed": seed}
    except Exception as e:
        return {"error": str(e)}

@app.get("/")
def root():
    return {"status": "ok", "model": "Qwen-Image-Edit-2511"}

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# Kill old processes
!fuser -k 8000/tcp || true
!pkill -f cloudflared || true
time.sleep(2)

# Download cloudflared if needed
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

threading.Thread(target=start_server, daemon=True).start()
print("Waiting for server to start...")
time.sleep(6)

print("Starting Cloudflare tunnel...")
proc = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(50):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*60)
    print("BACKEND READY")
    print(f"Paste this into the frontend Backend URL field:\n{public_url}")
    print("="*60)
    print(f"Test docs: {public_url}/docs")
else:
    print("Tunnel failed.")